In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.makedirs('/content/noongil', exist_ok=True)
os.chdir('/content/noongil')

In [3]:
!unzip /content/drive/MyDrive/NoonGil/noongil_colab_upload.zip -d /content/noongil

Archive:  /content/drive/MyDrive/NoonGil/noongil_colab_upload.zip
  inflating: /content/noongil/train_noongil.py  
   creating: /content/noongil/noongil_yolo/
  inflating: /content/noongil/noongil_yolo/.DS_Store  
   creating: /content/noongil/noongil_yolo/images/
   creating: /content/noongil/noongil_yolo/images/train/
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_PN000680.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_PN000657.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_B027530.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_B027524.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_PN000643.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_B027518.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_B027732.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_SEL_B027646.jpg  
  inflating: /content/noongil/noongil_yolo/images/train/MP_

In [4]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.9 MB/s eta 0:00:00


In [5]:
# 셀 6: train_noongil.py의 CONFIG project 경로를 Drive로 수정
import re

file_path = '/content/noongil/train_noongil.py'

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# "project" 키 값을 Drive 경로로 교체
content = re.sub(
    r'("project"\s*:\s*)["\'].*?["\']',
    r'\1"/content/drive/MyDrive/NoonGil/runs"',
    content
)

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ project 경로 수정 완료")
print("저장 경로: /content/drive/MyDrive/NoonGil/runs")

# 수정 확인
with open(file_path, 'r') as f:
    for i, line in enumerate(f, 1):
        if 'project' in line:
            print(f"  line {i}: {line.strip()}")

✅ project 경로 수정 완료
저장 경로: /content/drive/MyDrive/NoonGil/runs
  line 124: project=config["output_dir"],
  line 178: project=config["output_dir"],


In [7]:
import yaml
import os

def fix_yaml_paths(yaml_path, base_dir):
    """data.yaml의 train/val 경로를 Colab 절대경로로 수정"""
    if not os.path.exists(yaml_path):
        print(f"⚠️ {yaml_path} 없음, 스킵")
        return

    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # train/val 경로를 Colab 절대경로로 교체
    for key in ['train', 'val']:
        if key in data:
            # 경로에서 파일명/하위폴더만 추출해서 base_dir에 붙임
            rel = os.path.basename(data[key].rstrip('/'))
            data[key] = os.path.join(base_dir, 'images', rel)

    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, allow_unicode=True)

    print(f"✅ {yaml_path} 경로 수정 완료")
    print(f"   train: {data.get('train')}")
    print(f"   val:   {data.get('val')}")

# aihub_yolo/data.yaml 수정
fix_yaml_paths(
    '/content/noongil/aihub_yolo/data.yaml',
    '/content/noongil/aihub_yolo'
)

# noongil_yolo/data.yaml 수정
fix_yaml_paths(
    '/content/noongil/noongil_yolo/data.yaml',
    '/content/noongil/noongil_yolo'
)

✅ /content/noongil/aihub_yolo/data.yaml 경로 수정 완료
   train: /content/noongil/aihub_yolo/images/train
   val:   /content/noongil/aihub_yolo/images/val
✅ /content/noongil/noongil_yolo/data.yaml 경로 수정 완료
   train: /content/noongil/noongil_yolo/images/train
   val:   /content/noongil/noongil_yolo/images/val


In [9]:
import re

file_path = '/content/noongil/train_noongil.py'

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# device를 cuda(GPU)로 교체
content = re.sub(
    r'("device"\s*:\s*)["\'].*?["\']',
    r'\1"0"',
    content
)

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)

# 확인
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("✅ device 설정 완료 → GPU로 학습 실행")

CUDA 사용 가능: True
GPU: Tesla T4
✅ device 설정 완료 → GPU로 학습 실행


In [11]:
# Stage 2 best.pt 경로 확인
import glob

stage2_best = glob.glob('/content/noongil/runs/detect/runs/noongil/stage2_aihub*/weights/best.pt')
print("찾은 best.pt:", stage2_best)

# 가장 최근 best.pt 선택
if stage2_best:
    best_pt_path = sorted(stage2_best)[-1]
    print(f"✅ 사용할 모델: {best_pt_path}")
else:
    best_pt_path = 'yolov8s.pt'
    print("⚠️ best.pt 없음, 기본 모델 사용")

# Stage 3 직접 실행
from ultralytics import YOLO

model = YOLO(best_pt_path)
results = model.train(
    data='/content/noongil/noongil_yolo/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device='0',
    project='/content/drive/MyDrive/NoonGil/runs',
    name='stage3_noongil_final',
    patience=15,
)

찾은 best.pt: ['/content/noongil/runs/detect/runs/noongil/stage2_aihub-3/weights/best.pt']
✅ 사용할 모델: /content/noongil/runs/detect/runs/noongil/stage2_aihub-3/weights/best.pt
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/noongil/noongil_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

In [10]:
!python /content/noongil/train_noongil.py

Stage 1: COCO Pretrained YOLOv8 로드
✅ yolov8s.pt 로드 완료 (COCO 80 classes pretrained)
   - Backbone: CSPDarknet53
   - NoonGil 클래스와 겹치는 COCO 클래스 (전이학습 기대):
     → bicycle (COCO #1)
     → motorcycle (COCO #3)
     → fire_hydrant (COCO #10)
     → bench (COCO #13)
   - 직접 수집 필요 클래스 (COCO 없음):
     → trash_can, pothole, uneven_block, curb, ramp

Stage 2: AI Hub 데이터로 Fine-tuning
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./aihub_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez